# Coverage Audit Program

This program audits the coverage of dogwhistle terms in the benchmark dataset.
It calculates presence rates, type coverage, and identifies missing forms.

In [ ]:
# Imports
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import re


In [ ]:
@dataclass
class CoverageAuditConfig:
    # `workdir` is the anchor used to resolve all relative paths.
    workdir: Path

    # Local sources expected to exist in the repository.
    # Stage-06 glossary-mapped cleaned labels are the default audit input.
    data_local_path: Path = Path('../outputs/unioned_data/06_cleaned_labels_glossary_mapped.tsv')
    glossary_local_path: Path = Path('../outputs/unioned_data/06_glossary_label_reference.tsv')

    # Directory where audit exports are written.
    output_dir: Path = Path('../outputs/coverage_audits')

In [ ]:
# Configuration
WORKDIR = Path.cwd()
cfg = CoverageAuditConfig(workdir=WORKDIR)

# Convert all configured relative paths into absolute paths once up front.
cfg.output_dir = cfg.workdir / cfg.output_dir
cfg.data_path = cfg.workdir / cfg.data_local_path
cfg.glossary_path = cfg.workdir / cfg.glossary_local_path

# Create output/cache directories early so later cells can assume they exist.
cfg.output_dir.mkdir(parents=True, exist_ok=True)

cfg

In [ ]:
# Load data
data = pd.read_csv(cfg.data_path, sep='\t')

# Parse glossary with a fallback for spacing inconsistencies in delimiters.
glossary = pd.read_csv(cfg.glossary_path, sep='\t', engine='python')
glossary.columns = glossary.columns.str.strip().str.lower()
if not {'dogwhistle', 'surface_forms', 'taxonomy_level', 'type', 'target'}.issubset(glossary.columns):
    glossary = pd.read_csv(cfg.glossary_path, sep=r'\s{2,}|\t', engine='python')
    glossary.columns = glossary.columns.str.strip().str.lower()

required_glossary_cols = {'dogwhistle', 'surface_forms', 'taxonomy_level', 'type', 'target'}
missing_glossary_cols = required_glossary_cols - set(glossary.columns)
if missing_glossary_cols:
    raise ValueError(f"Glossary is missing required columns: {sorted(missing_glossary_cols)}")

# Normalize glossary fields.
glossary['dogwhistle'] = glossary['dogwhistle'].fillna('').astype(str).str.strip().str.lower()
glossary['surface_forms'] = glossary['surface_forms'].fillna('').astype(str)
glossary['target'] = glossary['target'].fillna('').astype(str).str.strip().str.lower()


def normalize_surface_forms(raw_surface_forms: str) -> list[str]:
    forms = [
        form.strip().lower()
        for form in str(raw_surface_forms).split(';')
        if form is not None and str(form).strip()
    ]
    base_forms = set(forms)
    normalized_forms = []

    for form in forms:
        # Some historical exports accidentally append a row/footnote number
        # (for example: "adult human females 4"). If the base form exists
        # in the same list, keep the base form and drop the numeric suffix.
        numeric_suffix_match = re.fullmatch(r'(.+?)\s+\d+', form)
        if numeric_suffix_match and numeric_suffix_match.group(1) in base_forms:
            form = numeric_suffix_match.group(1)
        normalized_forms.append(form)

    return normalized_forms


# Expand each dogwhistle into one row per surface form.
glossary_expanded = glossary.copy()
glossary_expanded['surface_form'] = glossary_expanded['surface_forms'].apply(normalize_surface_forms)
glossary_expanded = glossary_expanded.explode('surface_form')
glossary_expanded['surface_form'] = glossary_expanded['surface_form'].fillna('').astype(str).str.strip().str.lower()
glossary_expanded = glossary_expanded[glossary_expanded['surface_form'] != '']
glossary_expanded = glossary_expanded.drop_duplicates(
    ['dogwhistle', 'surface_form', 'taxonomy_level', 'target', 'type']
)

# Compile regex for all known surface forms.
all_forms = sorted(glossary_expanded['surface_form'].unique(), key=len, reverse=True)
pattern = re.compile(r'\b(' + '|'.join(map(re.escape, all_forms)) + r')\b', flags=re.IGNORECASE)


In [ ]:
# 1. Extract matched surface forms
data['found_forms'] = data['text'].apply(lambda x: pattern.findall(x.lower()) if pd.notna(x) else [])

# 2. Explode and join to glossary-expanded rows
matches_df = data.explode('found_forms').dropna(subset=['found_forms'])
matches_df['found_forms'] = matches_df['found_forms'].astype(str).str.strip().str.lower()

audit_df = matches_df.merge(
    glossary_expanded,
    left_on='found_forms',
    right_on='surface_form',
    how='inner'
 )

# Dedupe to one row per (post, dogwhistle) within level/target.
dogwhistle_hits_df = (
    audit_df.sort_values(['taxonomy_level', 'target', 'text_dedup_key', 'dogwhistle'])
    .drop_duplicates(['taxonomy_level', 'target', 'text_dedup_key', 'dogwhistle'])
    .copy()
)


In [ ]:
glossary_expanded

In [ ]:
# Build metrics aggregated by taxonomy_level and target
metrics_data = []

for (level, target), group in dogwhistle_hits_df.groupby(['taxonomy_level', 'target']):
    glossary_subset = glossary_expanded[
        (glossary_expanded['taxonomy_level'] == level) &
        (glossary_expanded['target'] == target)
    ]

    total_glossary_dogwhistles = glossary_subset['dogwhistle'].nunique()
    total_glossary_types = glossary_subset['type'].nunique()

    distinct_dogwhistles_found = group['dogwhistle'].nunique()
    distinct_types_found = group['type'].nunique()

    presence_rate = (
        distinct_dogwhistles_found / total_glossary_dogwhistles
        if total_glossary_dogwhistles > 0 else 0
    )
    type_coverage = distinct_types_found / total_glossary_types if total_glossary_types > 0 else 0

    # Hit count: one per (post, dogwhistle).
    weighted_hits = len(group)

    metrics_data.append({
        'taxonomy_level': level,
        'target': target,
        'total_glossary_dogwhistles': total_glossary_dogwhistles,
        'total_glossary_types': total_glossary_types,
        'distinct_dogwhistles_found': distinct_dogwhistles_found,
        'distinct_types_found': distinct_types_found,
        'presence_rate': presence_rate,
        'type_coverage': type_coverage,
        'token_frequency': weighted_hits
    })

final_report = pd.DataFrame(metrics_data)


In [ ]:
# Detailed breakdown: which dogwhistles appear in each level/target group
detailed_breakdown = []

for (level, target), group in dogwhistle_hits_df.groupby(['taxonomy_level', 'target']):
    for dogwhistle, dogwhistle_group in group.groupby('dogwhistle'):
        detailed_breakdown.append({
            'taxonomy_level': level,
            'target': target,
            'dogwhistle': dogwhistle,
            'matched_posts': dogwhistle_group['text_dedup_key'].nunique()
        })

detailed_report = pd.DataFrame(detailed_breakdown).sort_values(
    by=['taxonomy_level', 'target', 'matched_posts'], ascending=[True, True, False]
)

print("Detailed breakdown of dogwhistles found:")
print(detailed_report)

In [ ]:
# Summary: Missing dogwhistles (in glossary but not found in benchmark)
missing_dogwhistles = []

for (level, target), glossary_subset in glossary_expanded.groupby(['taxonomy_level', 'target']):
    glossary_dogwhistles = set(glossary_subset['dogwhistle'].unique())
    found_dogwhistles = set(
        detailed_report[
            (detailed_report['taxonomy_level'] == level) &
            (detailed_report['target'] == target)
        ]['dogwhistle'].unique()
    )
    missing = glossary_dogwhistles - found_dogwhistles

    for dogwhistle in sorted(missing):
        missing_dogwhistles.append({
            'taxonomy_level': level,
            'target': target,
            'dogwhistle': dogwhistle,
            'status': 'not_found'
        })

missing_report = pd.DataFrame(missing_dogwhistles)

print("=" * 80)
print("AUDIT SUMMARY")
print("=" * 80)
print(f"\nTotal benchmark posts analyzed: {len(data)}")
print(f"Total dogwhistle hits found: {len(dogwhistle_hits_df)}")
print(f"Unique dogwhistles found: {dogwhistle_hits_df['dogwhistle'].nunique()}")
print(f"\nTotal dogwhistles in glossary: {glossary['dogwhistle'].nunique()}")
print(f"Dogwhistles NOT found in benchmark: {len(missing_report)}")

print("\n" + "=" * 80)
print("METRICS BY TAXONOMY LEVEL & TARGET GROUP")
print("=" * 80)
print(final_report.to_string(index=False))

if len(missing_report) > 0:
    print("\n" + "=" * 80)
    print("DOGWHISTLES MISSING FROM BENCHMARK")
    print("=" * 80)
    print(missing_report.to_string(index=False))


In [ ]:
# Export results
audit_metrics_path = cfg.output_dir / 'audit_metrics.tsv'
audit_detailed_path = cfg.output_dir / 'audit_detailed.tsv'
audit_missing_path = cfg.output_dir / 'audit_missing.tsv'
audit_matches_path = cfg.output_dir / 'audit_matches.tsv'

final_report.to_csv(audit_metrics_path, sep='\t', index=False)
detailed_report.to_csv(audit_detailed_path, sep='\t', index=False)
if len(missing_report) > 0:
    missing_report.to_csv(audit_missing_path, sep='\t', index=False)

# Stage-01 pipeline input: row-level matched instances with dogwhistle IDs.
audit_matches_cols = [
    'text_dedup_key',
    'text',
    'binary_hate',
    'targets',
    'found_forms',
    'dogwhistle',
    'taxonomy_level',
    'target',
    'type'
 ]
audit_matches_report = (
    audit_df[audit_matches_cols]
    .drop_duplicates()
    .sort_values(['taxonomy_level', 'target', 'text_dedup_key', 'dogwhistle', 'found_forms'])
)
audit_matches_report.to_csv(audit_matches_path, sep='\t', index=False)

print(f"\nExported results:")
print(f"  - {audit_metrics_path}")
print(f"  - {audit_detailed_path}")
if len(missing_report) > 0:
    print(f"  - {audit_missing_path}")
print(f"  - {audit_matches_path}")